# Huffman algorithm

In [ ]:
import heapq
import random
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from functools import lru_cache
from itertools import combinations
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

#  Implementación Huffman Greedy

In [ ]:
def huffman_greedy(freqs):
    
    heap = freqs[:]

    heapq.heapify(heap)

    total_cost = 0

    while len(heap) > 1:

        x = heapq.heappop(heap)
        y = heapq.heappop(heap)

        z = x + y

        total_cost += z

        heapq.heappush(heap, z)

    return total_cost



# Explicación

El algoritmo greedy utiliza una cola de prioridad (heap).

En cada iteración:

1. Extrae las dos frecuencias más pequeñas.
2. Las combina.
3. Inserta nuevamente el nodo combinado.
4. Acumula el costo total.

Este proceso continúa hasta que queda un único nodo.

Complejidad:

$$
O(n \log n)
$$



# Implementación Huffman con Programación Dinámica

In [ ]:
def huffman_dp(freqs):
    
    freqs = tuple(sorted(freqs))

    @lru_cache(None)
    def solve(state):

        if len(state) == 1:
            return 0

        best = float('inf')

        for i, j in combinations(range(len(state)), 2):

            a = state[i]
            b = state[j]

            new_state = list(state)

            new_state.pop(j)
            new_state.pop(i)

            new_state.append(a + b)

            new_state = tuple(sorted(new_state))

            cost = solve(new_state) + a + b

            best = min(best, cost)

        return best

    return solve(freqs)

# Explicación

La solución con programación dinámica:

* Explora múltiples combinaciones posibles.
* Evalúa distintos árboles.
* Utiliza memoización para evitar recomputaciones.

Aunque garantiza encontrar el costo óptimo, su complejidad es mucho mayor.

Complejidad aproximada:

$$
O(2^n)
$$



# Pruebas

In [ ]:
freqs = [5, 9, 12, 13, 16, 45]

print("Frecuencias:", freqs)

print("Costo Huffman Greedy:", huffman_greedy(freqs))
print("Costo Huffman DP:", huffman_dp(freqs))

Se utiliza un conjunto pequeño de frecuencias para verificar:

Correctitud.
Comparación de resultados.

Ambos algoritmos deben producir el mismo costo óptimo

In [ ]:
sizes = [5, 6, 7, 8, 9, 10, 11, 12]

results = []

for n in sizes:

    freqs = [random.randint(1, 100) for _ in range(n)]

    # Tiempo DP
    start = time.perf_counter()
    dp_result = huffman_dp(freqs)
    dp_time = time.perf_counter() - start

    # Tiempo Greedy
    start = time.perf_counter()
    greedy_result = huffman_greedy(freqs)
    greedy_time = time.perf_counter() - start

    results.append({
        'n': n,
        'dp_time': dp_time,
        'greedy_time': greedy_time,
        'dp_cost': dp_result,
        'greedy_cost': greedy_result
    })

    

In [ ]:
df = pd.DataFrame(results)

print(df)

In [ ]:
print("¿Greedy produce solución óptima?")

print((df['dp_cost'] == df['greedy_cost']).all())

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(df['n'], df['dp_time'], label='DP')
plt.scatter(df['n'], df['greedy_time'], label='Greedy')

plt.xlabel('Tamaño de entrada')
plt.ylabel('Tiempo de ejecución (s)')
plt.title('DP vs Greedy - Huffman Coding')

plt.legend()
plt.grid(True)

plt.show()

# Regresión polinomial de Greedy

In [ ]:
X = df[['n']]
y = df['greedy_time']

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

model = LinearRegression()
model.fit(X_poly, y)

x_range = np.linspace(min(df['n']), max(df['n']), 100)
x_range_poly = poly.transform(x_range.reshape(-1,1))

y_pred = model.predict(x_range_poly)
plt.figure(figsize=(10,6))

plt.scatter(df['n'], df['greedy_time'])
plt.plot(x_range, y_pred)

plt.xlabel('Tamaño de entrada')
plt.ylabel('Tiempo Greedy')
plt.title('Regresión Polinomial - Greedy Huffman')

plt.grid(True)

plt.show()

# CSV

In [ ]:
df.to_csv('resultados_huffman.csv', index=False)

print('Archivo CSV generado correctamente')